# 03 — Train RL Agent v2 (LSTM + Attention + Dirichlet)

**WFO:** Sliding 24m train (non-anchored), 1m val, 1m test, 5-day embargo

**Checkpoint-resume:** Stop/restart kernel anytime — training resumes automatically.

Run `02_Run_Baselines.ipynb` first.

In [ ]:
import sys, os, time
import numpy as np
import pandas as pd
import torch

sys.path.insert(0, os.path.join(os.getcwd(), 'functions'))

print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'CUDA: {torch.cuda.get_device_name(0)}')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print('Device: MPS (Apple Silicon)')
else:
    print('Device: CPU')

In [ ]:
from data_pipeline import build_dataset
from RL_1.train import train_walk_forward, generate_wfo_folds, count_wfo_folds, plot_wfo_folds

dataset = build_dataset('../Data/Outputs/Filtered/Data')

## 1. Preview & Visualize WFO Folds

In [ ]:
wfo_info = count_wfo_folds(dataset['trading_dates'], train_months=24, val_months=1, test_months=1, step_months=1, embargo_days=5)
print(f'Total folds: {wfo_info["n_folds"]}')
if wfo_info['n_folds'] > 0:
    print(f'First test: {wfo_info["first_fold"]["test_start"]} → {wfo_info["first_fold"]["test_end"]}')
    print(f'Last test:  {wfo_info["last_fold"]["test_start"]} → {wfo_info["last_fold"]["test_end"]}')
    print(f'OOS: {wfo_info["total_test_period"][0]} → {wfo_info["total_test_period"][1]}')

In [ ]:
folds = generate_wfo_folds(dataset['trading_dates'], train_months=24, val_months=1, test_months=1, step_months=1, embargo_days=5)
fig = plot_wfo_folds(folds)
if fig: fig.show()

## 2. Train (Walk-Forward)
5 HP configs → sliding 24m window → checkpoint after each fold

In [ ]:
t0 = time.time()

rl_results = train_walk_forward(
    dataset,
    train_months=24,
    val_months=1,
    test_months=1,
    step_months=1,
    embargo_days=5,
    n_epochs=30,
    patience=5,
    min_epochs=10,
    transaction_cost_bps=5.0,
    turnover_penalty=0.001,
    variance_penalty=0.5,
    tc_curriculum_frac=0.3,
    lookback_window=60,
    results_dir='../Results',
    verbose=True,
)

print(f'\n\nTotal time: {(time.time()-t0)/60:.1f} minutes')

## 3. Results

In [ ]:
print('=' * 60)
print('OUT-OF-SAMPLE PERFORMANCE')
print('=' * 60)
rl_m = pd.read_csv('../Results/rl_performance_metrics.csv', index_col=0)
display(rl_m)

In [ ]:
fold_log = pd.read_csv('../Results/rl_fold_log.csv')
print(f'Retrains: {fold_log["retrained"].sum()} / {len(fold_log)} folds')
display(fold_log)

In [ ]:
bl_metrics = pd.read_csv('../Results/performance_metrics_full.csv', index_col=0)
rl_row = rl_m.loc[['RL Agent']]
combined = pd.concat([bl_metrics, rl_row]).sort_values('IR2', ascending=False)
display(combined[['ARC (%)', 'ASD (%)', 'Max Drawdown (%)', 'IR1', 'IR2']])

## 4. Files Saved

In [ ]:
print('RL files in ../Results/:')
for f in sorted(os.listdir('../Results')):
    if f.startswith('rl_') or f.startswith('agent_') or f.startswith('wfo_'):
        size = os.path.getsize(f'../Results/{f}') / 1024
        print(f'  {f:<45} {size:>8.1f} KB')